In [1]:
from unstructured.partition.pdf import partition_pdf
from unstructured.documents.elements import Text, Image, Table, ListItem, NarrativeText, Title, Address, PageBreak, FigureCaption

In [2]:
base_dir = "paper"
pdf_to_parse = "RAG-paper.pdf"

pdf_file_path = f"{base_dir}/{pdf_to_parse}"

In [3]:
raw_chunks = partition_pdf(
    filename=pdf_file_path,
    strategy="hi_res",
    infer_table_structure=True,
    extract_image_block_types=["figure", "Image", "Table"],
    extract_image_block_to_payload=True,
    chunking_strategy=None,
    )

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 995.08it/s, Materializing param=model.query_position_embeddings.weight]                             


In [4]:
raw_chunks

In [5]:
for idx, chunk in enumerate(raw_chunks):
    if isinstance(chunk, Image):
        print(f"Chunk {idx}: Image with metadata: {chunk.metadata}")
    

Chunk 27: Image with metadata: <unstructured.documents.elements.ElementMetadata object at 0x0000028B18CFA350>
Chunk 40: Image with metadata: <unstructured.documents.elements.ElementMetadata object at 0x0000028B18CFA850>
Chunk 54: Image with metadata: <unstructured.documents.elements.ElementMetadata object at 0x0000028B18CFAED0>
Chunk 299: Image with metadata: <unstructured.documents.elements.ElementMetadata object at 0x0000028B18E295D0>
Chunk 363: Image with metadata: <unstructured.documents.elements.ElementMetadata object at 0x0000028B18E144D0>
Chunk 470: Image with metadata: <unstructured.documents.elements.ElementMetadata object at 0x0000028B18DB99D0>


In [6]:
raw_chunks[27].to_dict()

{'type': 'Image',
 'element_id': 'a32f39aca7fe4ba9d96d81ad6d23c4be',
 'text': 'Fine-tuning UniMS-RAG PU PE 0 eascessesesssssssassssct hcg [passsssss2a5=ssss5=2= Dual-Feedback-ToD Self-RAG | MK-ToD LM-Indexer InstructRetro RAG_Robust RA-DIT Retrieve-and-Sample GPT-4 SANTA.) ( suGRE RRR Self-Mem UPRISE ~ 2023 =2~-~----------- ChatGPT GPT-3 21-2 teeta Retrieval—Augmented Generation Pre-training Filter-Reranker ICRALM Inference G-Retriever RADA RAPTOR DRAGON-AI CREA-ICL FILCO ARM-RAG PaperQA PRCA 1-PAGER ToC Token-Elimination FABULA KALMV QLM-Doc-ranking KCr KnowledGPT F LLM-R IRCOT PGRA PKG wy SCM4LLMs ITER-RETGEN \\ 5 Co! etro++ LUM Lite P GenRead Augmentation Stage Fine-tuning Pre-training Inference',
 'metadata': {'coordinates': {'points': ((np.float64(425.4347222222222),
     np.float64(272.51700000000017)),
    (np.float64(425.4347222222222), np.float64(1799.6222222222223)),
    (np.float64(2549.51375), np.float64(1799.6222222222223)),
    (np.float64(2549.51375), np.float64(272.5170

In [7]:
import base64
raw_chunks[28].to_dict()

{'type': 'FigureCaption',
 'element_id': '9feffca96f77383da082fe44e853b51d',
 'text': 'Fig. 1. Technology tree of RAG research. The stages of involving RAG mainly include pre-training, fine-tuning, and inference. With the emergence of LLMs, research on RAG initially focused on leveraging the powerful in context learning abilities of LLMs, primarily concentrating on the inference stage. Subsequent research has delved deeper, gradually integrating more with the fine-tuning of LLMs. Researchers have also been exploring ways to enhance language models in the pre-training stage through retrieval-augmented techniques.',
 'metadata': {'detection_class_prob': 0.8881842494010925,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(228.68211364746094),
     np.float64(1856.4215911111112)),
    (np.float64(228.68211364746094), np.float64(2025.9241605555555)),
    (np.float64(2758.593505859375), np.float64(2025.9241605555555)),
    (np.float64(2758.593505859375), np.float64(1856.421

In [8]:
all__image =[]

for idx, chunk in enumerate(raw_chunks):
    if isinstance(chunk, Image):
        # if idx+1 is figure caption, then add it to the image metadata

        if idx+1 < len(raw_chunks) and isinstance(raw_chunks[idx+1], FigureCaption):
            caption = raw_chunks[idx+1].text
        else:
            caption = None
        all__image.append({
            "index" : idx,
            "caption" : caption if caption else None,
            "image_text" : chunk.text,
            "base64_image" : chunk.metadata.image_base64,
        })

In [9]:
all__image[0]

{'index': 27,
 'caption': 'Fig. 1. Technology tree of RAG research. The stages of involving RAG mainly include pre-training, fine-tuning, and inference. With the emergence of LLMs, research on RAG initially focused on leveraging the powerful in context learning abilities of LLMs, primarily concentrating on the inference stage. Subsequent research has delved deeper, gradually integrating more with the fine-tuning of LLMs. Researchers have also been exploring ways to enhance language models in the pre-training stage through retrieval-augmented techniques.',
 'image_text': 'Fine-tuning UniMS-RAG PU PE 0 eascessesesssssssassssct hcg [passsssss2a5=ssss5=2= Dual-Feedback-ToD Self-RAG | MK-ToD LM-Indexer InstructRetro RAG_Robust RA-DIT Retrieve-and-Sample GPT-4 SANTA.) ( suGRE RRR Self-Mem UPRISE ~ 2023 =2~-~----------- ChatGPT GPT-3 21-2 teeta Retrieval—Augmented Generation Pre-training Filter-Reranker ICRALM Inference G-Retriever RADA RAPTOR DRAGON-AI CREA-ICL FILCO ARM-RAG PaperQA PRCA 1-P

In [10]:
import base64

from IPython.display import Image as IPImage
# Example: Display the first image in the notebook
def display_image(image_dict):
    image_data = base64.b64decode(image_dict["base64_image"])
    return IPImage(data=image_data)

In [11]:
# Image captioning using gemini-1.5-flash
from dotenv import load_dotenv
load_dotenv()
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [16]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)


def generate_caption(image_dict):
    prompt = (
        f"You are an assistant that generates a detailed description. the caption is: {image_dict['caption']}."
        f"The image text is: {image_dict['image_text']}."
        f"The description should be concise and descriptive, highlighting the main elements of the image."
        f"do not use any extraneous information that is not present in the image or the caption."
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_dict['base64_image']}"}},
                ],
            }
        ],
        max_tokens=300,
    )
    return response.choices[0].message.content

In [17]:
print(generate_caption(all__image[0]))

The image presents a technology tree for Retrieval-Augmented Generation (RAG) research, delineating the stages of Pre-training, Fine-tuning, and Inference. Each stage features a branching list of various models and methods, highlighting ongoing developments in the field.

1. **Fine-tuning (in green)**: 
   - Top entries include **UniMS-RAG**, **CoN**, **EAR**, and more, indicating advancements and refinements in model performance.
   - Other notable entries include **Dual-Feedback-ToD**, **Self-RAG**, and **Retrieve-and-Sample**.

2. **Pre-training (in orange)**: 
   - Key models like **InstructRetro**, **RAVEN**, and **CoG** are represented, showing a focus on foundational improvements in retrieval-augmented techniques.

3. **Inference (in blue)**: 
   - This section lists models such as **G-Retriever**, **RADA**, and **PaperQA**, signifying applications and performance evaluations post-training.
   - Additional models include **Itera-Retgen**, **KGP**, and **GenRead**.

The timeline 